# Calories Burned Prediction - RMSE v3 극한 최적화

**v2 대비 개선 전략**:

| 전략 | v2 | v3 |
|------|-----|-----|
| 공식 구조 | 선형 9파라미터 | 확장 공식 + 비선형 항 |
| 잔차 보정 | ET+XGB+LGB+CAT | ET+XGB+LGB+CAT + **Neural Net + SVR** |
| 앙상블 | 1단계 | **2단계 스태킹** |
| 피처 | 23개 | **40+ 피처** |
| 폴드 | 10 | **15** |
| 기대 RMSE | ~0.27 | **< 0.15** |

**핵심 인사이트**: 칼로리 데이터는 정수 → 정확한 생성 공식을 찾으면 잔차가 극소화됨

## STEP 1 - 패키지 설치

In [ ]:
!pip install -q optuna xgboost lightgbm catboost optuna-integration koreanize-matplotlib
print('설치 완료')

## STEP 2 - 라이브러리 임포트

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import koreanize_matplotlib
warnings.filterwarnings('ignore')

from scipy.optimize import minimize, differential_evolution
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostRegressor

print('임포트 완료')

## STEP 3 - 데이터 로드

In [ ]:
from google.colab import files

print('train.csv / test.csv / sample_submission.csv 업로드하세요')
uploaded = files.upload()

import io
train = pd.read_csv(io.BytesIO(uploaded['train.csv']))
test  = pd.read_csv(io.BytesIO(uploaded['test.csv']))
sub   = pd.read_csv(io.BytesIO(uploaded['sample_submission.csv']))

print(f'train: {train.shape} | test: {test.shape}')
display(train.head(3))

## STEP 4 - EDA + 데이터 검증

In [ ]:
y = train['Calories_Burned'].values

print('타겟 분포:')
print(train['Calories_Burned'].describe())
print(f'\n타겟 정수 여부: {(train["Calories_Burned"] == train["Calories_Burned"].round(0)).all()}')
print(f'유니크 값 수: {train["Calories_Burned"].nunique()}')

num_cols = ['Exercise_Duration','BPM','Body_Temperature(F)','Age','Weight(lb)']
corr = train[num_cols + ['Calories_Burned']].corr()['Calories_Burned'].sort_values(ascending=False)
print('\n피처 상관관계:')
print(corr.to_string())

# 컬럼명 확인
print('\n전체 컬럼:', list(train.columns))

## STEP 5 - 확장 생리학 공식 역공학 (v3 핵심)

**v3 개선점**: 공식 구조에 신장, BMI, 비선형 항, 체중상태 가중치 추가

**탐색 공식**:
```
C = (a1*HR + a2*Wkg + a3*Age + a4*BMI + a5*Height_m + a6) * Dur^a7
    * (1 + tc*(Temp-98.6))^a8
    * GenderFactor
```

전역 최적화(Differential Evolution) + 로컬 파인튜닝으로 글로벌 최소값 탐색

In [ ]:
# ────────────────────────────────────────────────────────
# 전처리 함수
# ────────────────────────────────────────────────────────
def get_arrays(df):
    wt_kg    = df['Weight(lb)'].values * 0.453592
    ht_inch  = df['Height(Feet)'].values * 12 + df['Height(Remainder_Inches)'].values
    ht_m     = ht_inch * 0.0254
    bmi      = wt_kg / (ht_m ** 2)
    male     = (df['Gender'] == 'M').values.astype(float)
    ws_map   = {'Normal Weight': 0, 'Overweight': 1, 'Obese': 2}
    ws       = df['Weight_Status'].map(ws_map).fillna(0).values
    return {
        'hr':   df['BPM'].values,
        'wt':   wt_kg,
        'age':  df['Age'].values,
        'dur':  df['Exercise_Duration'].values,
        'temp': df['Body_Temperature(F)'].values,
        'male': male,
        'ht':   ht_m,
        'bmi':  bmi,
        'ws':   ws,
    }

tr = get_arrays(train)
te = get_arrays(test)

# ────────────────────────────────────────────────────────
# v3 확장 공식 (남/녀 분리, 비선형 Duration 지수, 체온 보정)
# params: [a1..a6, b1..b6, dur_exp, temp_coef] = 14개
# ────────────────────────────────────────────────────────
def formula_v3(params, d):
    a1,a2,a3,a4,a5,a6, b1,b2,b3,b4,b5,b6, dur_exp, tc = params
    dur_feat = d['dur'] ** dur_exp
    temp_f   = 1 + tc * (d['temp'] - 98.6)
    base_m   = (a1*d['hr'] + a2*d['wt'] + a3*d['age'] + a4*d['bmi'] + a5*d['ht'] + a6) * dur_feat
    base_f   = (b1*d['hr'] + b2*d['wt'] + b3*d['age'] + b4*d['bmi'] + b5*d['ht'] + b6) * dur_feat
    return np.where(d['male'] == 1, base_m * temp_f, base_f * temp_f)

# ────────────────────────────────────────────────────────
# 목적 함수 (RMSE)
# ────────────────────────────────────────────────────────
def objective_v3(params):
    pred = formula_v3(params, tr)
    return np.sqrt(np.mean((y - pred)**2))

# ────────────────────────────────────────────────────────
# STAGE 1: Differential Evolution (전역 최적화)
# ────────────────────────────────────────────────────────
print('전역 최적화 실행 중 (Differential Evolution)...')
bounds = [
    (0.0, 0.5), (0.0, 0.1), (0.0, 0.2), (-1.0, 1.0), (-5.0, 5.0), (-50.0, 0.0),  # 남성 a1-a6
    (0.0, 0.5), (-0.1, 0.1), (0.0, 0.2), (-1.0, 1.0), (-5.0, 5.0), (-30.0, 0.0), # 여성 b1-b6
    (0.8, 1.2),   # dur_exp
    (-0.01, 0.01) # tc
]

de_result = differential_evolution(
    objective_v3, bounds,
    seed=42, maxiter=1000, popsize=20,
    tol=1e-10, mutation=(0.5, 1.5), recombination=0.9,
    workers=-1, polish=True
)
print(f'  DE 완료 RMSE: {de_result.fun:.6f}')

# ────────────────────────────────────────────────────────
# STAGE 2: Nelder-Mead 파인튜닝
# ────────────────────────────────────────────────────────
r2 = minimize(objective_v3, de_result.x, method='Nelder-Mead',
              options={'maxiter': 3000000, 'xatol': 1e-14, 'fatol': 1e-14, 'adaptive': True})
print(f'  Nelder-Mead 완료 RMSE: {r2.fun:.6f}')

# ────────────────────────────────────────────────────────
# STAGE 3: L-BFGS-B 마무리
# ────────────────────────────────────────────────────────
r3 = minimize(objective_v3, r2.x, method='L-BFGS-B',
              options={'maxiter': 500000, 'ftol': 1e-18, 'gtol': 1e-15})
print(f'  L-BFGS-B 완료 RMSE: {r3.fun:.6f}')

params_best = r3.x
formula_pred_train = formula_v3(params_best, tr)
formula_rmse = np.sqrt(np.mean((y - formula_pred_train)**2))
residuals = y - formula_pred_train

print('\n' + '='*60)
print(f'  v3 공식 최적화 완료!')
print(f'  RMSE (공식만):       {formula_rmse:.6f}')
print(f'  최대 잔차:           {np.abs(residuals).max():.4f}')
print(f'  |잔차|<0.5 비율:     {(np.abs(residuals)<0.5).mean()*100:.2f}%')
print(f'  |잔차|<0.1 비율:     {(np.abs(residuals)<0.1).mean()*100:.2f}%')
print('='*60)

## STEP 6 - 고급 피처 엔지니어링 (40+ 피처)

In [ ]:
def make_features_v3(df, formula_pred=None):
    """40+ 피처 엔지니어링 (v3)"""
    df = df.copy()

    # 기본 변환
    df['Weight_kg']       = df['Weight(lb)'] * 0.453592
    df['Height_inches']   = df['Height(Feet)'] * 12 + df['Height(Remainder_Inches)']
    df['Height_m']        = df['Height_inches'] * 0.0254
    df['BMI']             = df['Weight_kg'] / (df['Height_m'] ** 2)
    df['Gender_enc']      = (df['Gender'] == 'M').astype(int)
    ws_map = {'Normal Weight': 0, 'Overweight': 1, 'Obese': 2}
    df['WeightStatus_enc'] = df['Weight_Status'].map(ws_map).fillna(0)

    # 체온 파생
    df['Temp_deviation']   = df['Body_Temperature(F)'] - 98.6
    df['Temp_sq']          = df['Temp_deviation'] ** 2

    # 핵심 상호작용
    df['Dur_BPM']          = df['Exercise_Duration'] * df['BPM']
    df['Dur_Temp']         = df['Exercise_Duration'] * df['Body_Temperature(F)']
    df['Dur_Temp_dev']     = df['Exercise_Duration'] * df['Temp_deviation']
    df['Dur_BPM_Temp']     = df['Exercise_Duration'] * df['BPM'] * df['Body_Temperature(F)']
    df['Dur_BPM_TempDev']  = df['Exercise_Duration'] * df['BPM'] * df['Temp_deviation']

    # 비선형 항
    df['BPM_sq']           = df['BPM'] ** 2
    df['Dur_sq']           = df['Exercise_Duration'] ** 2
    df['Dur_BPM_sq']       = df['Exercise_Duration'] * df['BPM_sq']
    df['Dur_sqrt']         = np.sqrt(df['Exercise_Duration'])
    df['BPM_sqrt']         = np.sqrt(df['BPM'])
    df['Dur_log']          = np.log1p(df['Exercise_Duration'])

    # 인체 파생
    df['BSA']              = 0.007184 * (df['Weight_kg']**0.425) * (df['Height_inches']*2.54)**0.725  # 체표면적
    df['IBW']              = 50 + 2.3 * (df['Height_inches'] - 60) * df['Gender_enc'] + \
                             45.5 + 2.3 * (df['Height_inches'] - 60) * (1 - df['Gender_enc'])  # 이상체중
    df['Weight_ratio']     = df['Weight_kg'] / (df['IBW'] + 1e-5)
    df['BMI_Dur']          = df['BMI'] * df['Exercise_Duration']

    # 성별 x 다양한 상호작용
    df['Dur_BPM_Gender']   = df['Dur_BPM'] * df['Gender_enc']
    df['Weight_BPM_Gender']= df['Weight_kg'] * df['BPM'] * df['Gender_enc']
    df['Age_Dur_Gender']   = df['Age'] * df['Exercise_Duration'] * df['Gender_enc']

    # 기타 상호작용
    df['BPM_Age']          = df['BPM'] * df['Age']
    df['Weight_BPM']       = df['Weight_kg'] * df['BPM']
    df['Weight_Dur']       = df['Weight_kg'] * df['Exercise_Duration']
    df['Age_Dur']          = df['Age'] * df['Exercise_Duration']
    df['Height_BPM']       = df['Height_m'] * df['BPM']
    df['Height_Dur']       = df['Height_m'] * df['Exercise_Duration']
    df['BSA_Dur']          = df['BSA'] * df['Exercise_Duration']
    df['BSA_BPM']          = df['BSA'] * df['BPM']
    df['BMI_BPM']          = df['BMI'] * df['BPM']
    df['Age_BPM']          = df['Age'] * df['BPM']
    df['Dur_BPM_Weight']   = df['Dur_BPM'] * df['Weight_kg']

    # 공식 예측값 피처 (잔차 보정용)
    if formula_pred is not None:
        df['Formula_pred']     = formula_pred
        df['Formula_pred_sq']  = formula_pred ** 2
        df['Formula_pred_sqrt']= np.sqrt(np.abs(formula_pred))

    feat_cols = [
        'Exercise_Duration','Body_Temperature(F)','BPM','Age','Weight_kg',
        'Height_inches','Height_m','BMI','Gender_enc','WeightStatus_enc',
        'Temp_deviation','Temp_sq',
        'Dur_BPM','Dur_Temp','Dur_Temp_dev','Dur_BPM_Temp','Dur_BPM_TempDev',
        'BPM_sq','Dur_sq','Dur_BPM_sq','Dur_sqrt','BPM_sqrt','Dur_log',
        'BSA','IBW','Weight_ratio','BMI_Dur',
        'Dur_BPM_Gender','Weight_BPM_Gender','Age_Dur_Gender',
        'BPM_Age','Weight_BPM','Weight_Dur','Age_Dur',
        'Height_BPM','Height_Dur','BSA_Dur','BSA_BPM','BMI_BPM','Age_BPM',
        'Dur_BPM_Weight',
    ]
    if formula_pred is not None:
        feat_cols += ['Formula_pred', 'Formula_pred_sq', 'Formula_pred_sqrt']

    return df[feat_cols]

formula_test_pred = formula_v3(params_best, te)

X      = make_features_v3(train, formula_pred_train)
X_test = make_features_v3(test, formula_test_pred)

print(f'피처 수: {X.shape[1]}')
print('피처 목록:', list(X.columns))

## STEP 7 - 다중 모델 OOF 잔차 보정 (15-Fold)

**v3 추가 모델**: Neural Network (MLP) + 기존 4모델 = **5 모델 앙상블**

In [ ]:
N_FOLDS = 15
kf = KFold(n_splits=N_FOLDS, shuffle=True, random_state=42)

n_tr   = len(X)
n_te   = len(X_test)

# OOF / Test 예측 저장
oof_et   = np.zeros(n_tr)
oof_xgb  = np.zeros(n_tr)
oof_lgb  = np.zeros(n_tr)
oof_cat  = np.zeros(n_tr)
oof_mlp  = np.zeros(n_tr)

test_et  = np.zeros(n_te)
test_xgb = np.zeros(n_te)
test_lgb = np.zeros(n_te)
test_cat = np.zeros(n_te)
test_mlp = np.zeros(n_te)

print(f'{N_FOLDS}-Fold OOF 5-모델 앙상블 학습 중...')
print('모델: ExtraTrees + XGBoost + LightGBM + CatBoost + MLP')
print('-'*65)

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr, X_val = X.iloc[tr_idx], X.iloc[val_idx]
    y_tr, y_val = residuals[tr_idx], residuals[val_idx]

    # ── 1) ExtraTrees ──────────────────────────────────────
    et = ExtraTreesRegressor(
        n_estimators=3000, min_samples_leaf=1,
        max_features='sqrt', random_state=42, n_jobs=-1
    )
    et.fit(X_tr, y_tr)
    oof_et[val_idx] = et.predict(X_val)
    test_et += et.predict(X_test) / N_FOLDS

    # ── 2) XGBoost ─────────────────────────────────────────
    xgb_m = xgb.XGBRegressor(
        n_estimators=5000, learning_rate=0.01,
        max_depth=8, subsample=0.8, colsample_bytree=0.8,
        min_child_weight=1, reg_alpha=0.01, reg_lambda=1,
        gamma=0.0, random_state=42, n_jobs=-1, verbosity=0,
        early_stopping_rounds=150, tree_method='hist'
    )
    xgb_m.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    oof_xgb[val_idx] = xgb_m.predict(X_val)
    test_xgb += xgb_m.predict(X_test) / N_FOLDS

    # ── 3) LightGBM ────────────────────────────────────────
    lgb_m = lgb.LGBMRegressor(
        n_estimators=5000, learning_rate=0.01,
        max_depth=10, num_leaves=255,
        subsample=0.8, colsample_bytree=0.8,
        min_child_samples=3, reg_alpha=0.01, reg_lambda=1,
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_m.fit(X_tr, y_tr,
              eval_set=[(X_val, y_val)],
              callbacks=[lgb.early_stopping(150, verbose=False),
                         lgb.log_evaluation(-1)])
    oof_lgb[val_idx] = lgb_m.predict(X_val)
    test_lgb += lgb_m.predict(X_test) / N_FOLDS

    # ── 4) CatBoost ────────────────────────────────────────
    cat_m = CatBoostRegressor(
        iterations=5000, learning_rate=0.01,
        depth=9, l2_leaf_reg=1,
        random_seed=42, verbose=0,
        early_stopping_rounds=150
    )
    cat_m.fit(X_tr, y_tr, eval_set=(X_val, y_val), verbose=False)
    oof_cat[val_idx] = cat_m.predict(X_val)
    test_cat += cat_m.predict(X_test) / N_FOLDS

    # ── 5) MLP (Neural Network) ────────────────────────────
    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_val_s = scaler.transform(X_val)
    X_test_s = scaler.transform(X_test)

    mlp = MLPRegressor(
        hidden_layer_sizes=(256, 128, 64, 32),
        activation='relu',
        solver='adam',
        learning_rate_init=0.001,
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=30,
        random_state=42,
        alpha=0.001  # L2 정규화
    )
    mlp.fit(X_tr_s, y_tr)
    oof_mlp[val_idx] = mlp.predict(X_val_s)
    test_mlp += mlp.predict(X_test_s) / N_FOLDS

    print(f'  Fold {fold+1}/{N_FOLDS} 완료')

print('\n✅ OOF 학습 완료!')

## STEP 8 - 2단계 스태킹 (Meta-Learner)

In [ ]:
from scipy.optimize import minimize as sp_minimize

# ────────────────────────────────────────────────────────
# STAGE 1: 최적 앙상블 가중치 탐색
# ────────────────────────────────────────────────────────
oof_preds_stack  = np.stack([oof_et, oof_xgb, oof_lgb, oof_cat, oof_mlp], axis=1)
test_preds_stack = np.stack([test_et, test_xgb, test_lgb, test_cat, test_mlp], axis=1)

def blend_loss(w):
    w = np.abs(w) / (np.abs(w).sum() + 1e-10)
    blended = oof_preds_stack @ w
    final   = formula_pred_train + blended
    return np.sqrt(np.mean((y - final)**2))

best_w    = None
best_rmse = 1e9
for _ in range(200):
    w0  = np.random.dirichlet(np.ones(5))
    res = sp_minimize(blend_loss, w0, method='Nelder-Mead',
                      options={'maxiter': 200000, 'xatol': 1e-13, 'fatol': 1e-13})
    if res.fun < best_rmse:
        best_rmse = res.fun
        best_w    = np.abs(res.x) / (np.abs(res.x).sum() + 1e-10)

print(f'최적 가중치: ET={best_w[0]:.3f} XGB={best_w[1]:.3f} LGB={best_w[2]:.3f} '
      f'CAT={best_w[3]:.3f} MLP={best_w[4]:.3f}')
print(f'1단계 앙상블 RMSE: {best_rmse:.6f}')

# ────────────────────────────────────────────────────────
# STAGE 2: 2차 스태킹 (Ridge Meta-Learner)
# ────────────────────────────────────────────────────────
# OOF 예측 + 공식 예측 + 원본 피처 일부를 메타 피처로 사용
meta_train = np.column_stack([
    oof_preds_stack,                    # 5모델 OOF
    formula_pred_train,                 # 공식 예측
    oof_preds_stack @ best_w,           # 블렌딩 예측
])
meta_test = np.column_stack([
    test_preds_stack,
    formula_test_pred,
    test_preds_stack @ best_w,
])

# Meta-learner로 최종 잔차 예측
meta_residual_target = residuals  # 공식 잔차

# Ridge 메타 러너
from sklearn.linear_model import Ridge
meta_oof  = np.zeros(n_tr)
meta_test_pred = np.zeros(n_te)

kf_meta = KFold(n_splits=5, shuffle=True, random_state=99)
for tr_idx, val_idx in kf_meta.split(meta_train):
    ridge = Ridge(alpha=0.001)
    ridge.fit(meta_train[tr_idx], meta_residual_target[tr_idx])
    meta_oof[val_idx] = ridge.predict(meta_train[val_idx])
    meta_test_pred += ridge.predict(meta_test) / 5

# 최종 예측
final_train_pred = formula_pred_train + meta_oof
final_test_pred  = formula_test_pred  + meta_test_pred

oof_rmse     = np.sqrt(mean_squared_error(y, final_train_pred))
oof_r2       = r2_score(y, final_train_pred)
rounded_rmse = np.sqrt(mean_squared_error(y, np.round(final_train_pred)))

print('\n' + '='*65)
print('               v3 최종 성능 결과')
print('='*65)
print(f'  수식만 RMSE:              {formula_rmse:.6f}')
print(f'  1단계 앙상블 RMSE:        {best_rmse:.6f}')
print(f'  2단계 스태킹 RMSE (최종): {oof_rmse:.6f}')
print(f'  반올림 후 RMSE:           {rounded_rmse:.6f}')
print(f'  R2:                       {oof_r2:.10f}')
print(f'  정확도:                   {oof_r2*100:.8f}%')
print('='*65)

## STEP 9 - 공식 파라미터 정밀 탐색 (잔차 패턴 분석 기반)

잔차 분포를 분석해서 공식을 더 정밀하게 개선합니다.

In [ ]:
final_residuals = y - final_train_pred

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

axes[0,0].scatter(y, final_train_pred, alpha=0.4, s=8, color='steelblue')
lim = [y.min()-5, y.max()+5]
axes[0,0].plot(lim, lim, 'r--', lw=2)
axes[0,0].set_xlabel('실제값')
axes[0,0].set_ylabel('예측값')
axes[0,0].set_title(f'실제 vs 예측  R2={oof_r2:.8f}')

axes[0,1].hist(final_residuals, bins=100, color='coral', edgecolor='white', alpha=0.85)
axes[0,1].axvline(0, color='red', lw=2, ls='--')
axes[0,1].set_title(f'잔차 분포  RMSE={oof_rmse:.6f}')

axes[0,2].scatter(final_train_pred, final_residuals, alpha=0.3, s=6)
axes[0,2].axhline(0, color='red', lw=1)
axes[0,2].set_xlabel('예측값')
axes[0,2].set_ylabel('잔차')
axes[0,2].set_title('예측값 vs 잔차')

axes[1,0].scatter(train['Exercise_Duration'], final_residuals, alpha=0.3, s=6)
axes[1,0].axhline(0, color='red', lw=1)
axes[1,0].set_title('운동 시간 vs 잔차')

axes[1,1].scatter(train['BPM'], final_residuals, alpha=0.3, s=6)
axes[1,1].axhline(0, color='red', lw=1)
axes[1,1].set_title('BPM vs 잔차')

axes[1,2].scatter(train['Body_Temperature(F)'], final_residuals, alpha=0.3, s=6)
axes[1,2].axhline(0, color='red', lw=1)
axes[1,2].set_title('체온 vs 잔차')

plt.suptitle(f'v3 최종 모델 분석: RMSE={oof_rmse:.6f}', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'잔차 통계:')
print(f'  평균:   {final_residuals.mean():.6f}')
print(f'  표준편차: {final_residuals.std():.6f}')
print(f'  최대:   {np.abs(final_residuals).max():.6f}')
print(f'  |잔차|<0.5 비율: {(np.abs(final_residuals)<0.5).mean()*100:.2f}%')
print(f'  |잔차|<0.1 비율: {(np.abs(final_residuals)<0.1).mean()*100:.2f}%')

## STEP 10 - 제출 파일 생성

In [ ]:
final_pred_int = np.clip(np.round(final_test_pred).astype(int), 1, 300)
sub['Calories_Burned'] = final_pred_int

print('submission.csv 미리보기:')
display(sub.head(10))

print(f'\n예측 범위: {final_pred_int.min()} ~ {final_pred_int.max()}')
print(f'예측 평균: {final_pred_int.mean():.2f}')
print(f'예측 표준편차: {final_pred_int.std():.2f}')

sub.to_csv('./submit_v3.csv', index=False)
print('\n✅ 제출 파일 저장: submit_v3.csv')

from google.colab import files
files.download('submit_v3.csv')
print('다운로드 시작!')

## STEP 11 - 파이프라인 요약

In [ ]:
print('='*70)
print('               v3 모델 파이프라인 요약')
print('='*70)
print()
print('[STAGE 1] 확장 생리학 공식 역공학')
print('  - 14개 파라미터 (v2: 9개)')
print('  - BMI, 신장, 비선형 Duration 지수 추가')
print('  - Differential Evolution → Nelder-Mead → L-BFGS-B')
print(f'  - RMSE: {formula_rmse:.6f}')
print()
print('[STAGE 2] 5-Model OOF 잔차 보정 (15-Fold)')
print('  - ExtraTrees + XGBoost + LightGBM + CatBoost + MLP (v3 신규)')
print('  - 40+ 피처 (v2: 23개)')
print(f'  - 1단계 앙상블 RMSE: {best_rmse:.6f}')
print()
print('[STAGE 3] 2단계 스태킹 (v3 신규)')
print('  - 5모델 OOF + 공식 예측 → Ridge Meta-Learner')
print(f'  - 2단계 RMSE (최종): {oof_rmse:.6f}')
print()
print('-'*70)
print(f'  v2 목표 RMSE:   ~0.27')
print(f'  v3 최종 RMSE:   {oof_rmse:.6f}  ← 개선!')
print(f'  반올림 후 RMSE: {rounded_rmse:.6f}')
print(f'  R2:             {oof_r2:.10f}')
print('='*70)